## Preprocess Data

In [2]:
# %%writefile ../build/movies_preprocess.py

"""
movies_preprocess.py
---------------------
Preprocess MovieLens 100k dataset and build FAISS index.
"""

import pandas as pd


def build_movies_clean_csv():
    # --- Step 1: Load raw MovieLens files ---
    movies = pd.read_csv(
        "../data/ml-100k/u.item",
        sep="|", encoding="latin-1",
        names=["movie_id","title","release_date","video_release_date","IMDb_URL",
               "unknown","Action","Adventure","Animation","Children","Comedy","Crime",
               "Documentary","Drama","Fantasy","Film-Noir","Horror","Musical","Mystery",
               "Romance","Sci-Fi","Thriller","War","Western"]
    )

    ratings = pd.read_csv(
        "../data/ml-100k/u.data",
        sep="\t",
        names=["user_id","item_id","rating","timestamp"]
    )

    # --- Step 2: Genres ---
    genre_cols = [
        "Action","Adventure","Animation","Children","Comedy","Crime",
        "Documentary","Drama","Fantasy","Film-Noir","Horror","Musical",
        "Mystery","Romance","Sci-Fi","Thriller","War","Western"
    ]

    movies["genre_list"] = movies.apply(
        lambda row: ", ".join([g for g in genre_cols if row[g] == 1]),
        axis=1
    )

    # --- Step 3: Ratings summary ---
    ratings_summary = ratings.groupby("item_id").agg(
        avg_rating=("rating", "mean"),
        rating_count=("rating", "count")
    ).reset_index().rename(columns={"item_id": "movie_id"})

    movies_enriched = movies.merge(ratings_summary, on="movie_id", how="left")

    # --- Step 4: Final dataset ---
    movies_final = movies_enriched[
        ["movie_id", "title", "release_date", "genre_list", "avg_rating", "rating_count"]
    ]

    print("Processed movies:", len(movies_final))

    # --- Step 5: Save cleaned CSV ---
    movies_final.to_csv(
        "../cleaned_data/movies_clean.csv",
        index=False
    )

    print("✅ Cleaned movies dataset saved at ../cleaned_data/movies_clean.csv")

    return movies_final


if __name__ == "__main__":
    build_movies_clean_csv()

Writing ../build/movies_preprocess.py


In [1]:
%%writefile ../build/movies_index_build.py

"""
movies_index_build.py
--------------------
Load cleaned movies dataset,
generate embeddings,
and build FAISS index.
"""

import pandas as pd

from langchain_core.documents import Document
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
import os

def build_movies_index():

    movies = pd.read_csv("../cleaned_data/movies_clean.csv")
    # --- Step 5: Convert rows into LangChain Documents ---
    movie_docs = []

    for _, row in movies.iterrows():
        text = f"""
        Title: {row['title']}. 
        Released in {row['release_date']}. 
        Genres include {row['genre_list']}. 
        It has an average rating of {row['avg_rating']} based on {row['rating_count']} user ratings.
        """
    
        text = " ".join(text.split())
    
        movie_docs.append(Document(page_content=text))
    
    print(f"✅ Prepared {len(movie_docs)} movie documents.")

    # --- Step 6: Build embeddings + FAISS index ---
    embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
    vectorstore = FAISS.from_documents(movie_docs, embeddings)

    # --- Step 7: Save index ---
    output_path = "../vectorstores/faiss_movies_index"
    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    vectorstore.save_local(output_path)

    
    print("✅ Movies FAISS index built and saved at ../data/faiss_movies_index")

if __name__ == "__main__":
    build_movies_index()

Overwriting ../build/movies_index_build.py


## Validate Data

In [ ]:
# Total rows
total_rows = len(movies_final)

# Count missing and non-missing per column
missing_count = movies_final.isnull().sum()
non_missing_count = total_rows - missing_count
missing_percent = (missing_count / total_rows) * 100

# Combine into one DataFrame
missing_summary = pd.DataFrame({
    "non_missing": non_missing_count,
    "missing": missing_count,
    "missing_percent": missing_percent.round(2)
})

print(missing_summary)

## Load Variables

In [6]:
# load API key from .env 
from dotenv import load_dotenv, find_dotenv 
load_dotenv(find_dotenv())
# llm model
model="gpt-4o-mini"

## Business Logic

In [8]:
# %%writefile ../src/chain_movies.py
"""
chain_movies.py
----------------
Core LangChain module for MoviesRAG.

Responsibilities:
- Load the persisted FAISS index (built once in build/movies_build.py)
- Define a retrieval-augmented generation (RAG) chain for movie queries
- Designed for integration into the CultRAG pipeline
"""

# --- BUSINESS LOGIC ---

# Core LangChain modules
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate

# Embeddings + Vectorstore
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from utils.paths import VECTORSTORES_DIR

# Step 1: Load persisted FAISS index (built once in build/movies_build.py)
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
vectorstore = FAISS.load_local(
    str(VECTORSTORES_DIR / "faiss_movies_index"),
    embeddings,
    allow_dangerous_deserialization=True
)
retriever = vectorstore.as_retriever()

# Step 2: LLM → define the language model
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# Step 3: Prompt Template → instructions for how the assistant should answer
prompt_movies = ChatPromptTemplate.from_template("""
You are a movie recommendation assistant using retrieval-augmented generation (RAG).

Use ONLY the provided context to answer the question.
If the answer is not in the context, say "Not found in the catalog."

Conversation so far:
{history}

Context:
{context}

Question:
{question}

Rules:
- ONLY include MOVIES in your answer.
- DO NOT include books, songs, or any other media, even if present in the context.
- Ignore any non-movie entries in the context completely.
- Do NOT add items that are not in the context.
- Do NOT guess or hallucinate.
- Do NOT mention other domains (books, songs) or their absence.
- Output a valid markdown table with headers.
- Include a short summary after the table.

Answer:
""")

# Step 4: Helper to format retrieved docs into a single string
def format_docs(docs):
    """
    Convert a list of LangChain Document objects into a single string.
    Each document’s page_content is joined with double newlines.
    Used to feed retrieved context into the prompt.
    """
    return "\n\n".join([d.page_content for d in docs])

# Step 5: Build the Core LCEL Retrieval Chain
chain_movies = (
    {
        "context": lambda x: format_docs(retriever.invoke(x["question"])),
        "question": lambda x: x["question"],
        "history": lambda x: x.get("history", "")
    }
    | prompt_movies
    | llm
)


Overwriting ../src/chain_movies.py


## Query & Display

In [ ]:
# Display helpers for Jupyter
from IPython.display import display, Markdown

# --- 8. Query ---
query = "Please list 3 good sci fi movies"

response = chain_movies.invoke({"question": query})

# --- 9. Display ---
display(Markdown(response.content))